# US sector ETF rolling active mean-variance

Long-only 1x sector ETF allocation. This notebook is restricted to 2010-2019 in-sample data and does not inspect 2020 onward. Each week, use the previous 3 months of daily returns to solve for sector weights that maximize active utility relative to 100% SPY:

`mean(portfolio - SPY) - 0.5 * RISK_AVERSION * variance(portfolio - SPY)`

Weights are constrained to be long-only and sum to 1. The portfolio is held for one week, then optimized again.

For each rebalance date, the optimizer forms active returns for each sector: `sector_return - SPY_return`. It estimates the active mean vector `mu` and covariance matrix `Sigma` from the trailing window, then solves `max_w w @ mu - 0.5 * RISK_AVERSION * w @ Sigma @ w`, subject to `w >= 0` and `sum(w) = 1`.

In [ ]:
import pandas as pd

from alpha_lab.analytics.returns import cumulative_returns
from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.portfolio.active_mv import rolling_active_mean_variance_weights
from alpha_lab.reporting.charts import drawdown_chart, equity_curve, heatmap_monthly
from alpha_lab.utils.paths import CONFIGS_DIR

In [ ]:
TRAIN_START = "2010-01-01"
TRAIN_END = "2019-12-31"

BENCHMARK = "SPY"
LOOKBACK_DAYS = 63
REBALANCE = "W-FRI"
RISK_AVERSION = 1.0

COSTS_BPS = 1.0
SLIPPAGE_BPS = 2.0

In [ ]:
universe = pd.read_csv(CONFIGS_DIR / "us_sector_etf.csv").fillna("")
sector_tickers = universe["long_1x_etf"].tolist()
universe[["sector", "long_1x_etf"]]

In [ ]:
prices = load_prices(sector_tickers + [BENCHMARK], start=TRAIN_START, end=TRAIN_END)
sector_prices = prices[sector_tickers]
benchmark_prices = prices[BENCHMARK]
prices.loc[TRAIN_START:TRAIN_END].tail()

In [ ]:
pd.Series(
    {
        "first_price_date": prices.index.min().date(),
        "last_price_date": prices.index.max().date(),
        "n_price_rows": len(prices),
        "has_2020_or_later": bool((prices.index >= "2020-01-01").any()),
    }
)

In [ ]:
weights = rolling_active_mean_variance_weights(
    sector_prices,
    benchmark_prices,
    lookback_days=LOOKBACK_DAYS,
    rebalance=REBALANCE,
    risk_aversion=RISK_AVERSION,
)
weights.tail()

In [ ]:
res = run_backtest(
    weights,
    sector_prices,
    rebalance=REBALANCE,
    costs_bps=COSTS_BPS,
    slippage_bps=SLIPPAGE_BPS,
)
benchmark_returns = benchmark_prices.pct_change().reindex(res.returns.index).fillna(0.0)

first_invested = res.weights.abs().sum(axis=1).gt(0).idxmax()
perf_returns = pd.DataFrame(
    {
        "strategy": res.returns.loc[first_invested:],
        "SPY buy&hold": benchmark_returns.loc[first_invested:],
    }
).dropna()
active_returns = perf_returns["strategy"] - perf_returns["SPY buy&hold"]

pd.DataFrame(
    {
        "strategy": pd.Series(summary(perf_returns["strategy"])),
        "SPY buy&hold": pd.Series(summary(perf_returns["SPY buy&hold"])),
    }
)

In [ ]:
equity_curve(perf_returns)

In [ ]:
strategy_wealth = cumulative_returns(perf_returns["strategy"])
spy_wealth = cumulative_returns(perf_returns["SPY buy&hold"])
relative_wealth = strategy_wealth / spy_wealth

active_stats = pd.Series(
    {
        "Start": perf_returns.index[0].date(),
        "End": perf_returns.index[-1].date(),
        "Final strategy wealth": strategy_wealth.iloc[-1],
        "Final SPY wealth": spy_wealth.iloc[-1],
        "Final relative wealth": relative_wealth.iloc[-1],
        "Annualized active return": active_returns.mean() * 252,
        "Tracking error": active_returns.std() * (252**0.5),
        "Information ratio": active_returns.mean() / active_returns.std() * (252**0.5),
        "Active hit rate": (active_returns > 0).mean(),
    }
)
active_stats

In [ ]:
relative_wealth.rename("strategy / SPY wealth").to_frame().plot(title="Relative wealth vs SPY")

In [ ]:
rolling_active_1y = (1 + perf_returns["strategy"]).rolling(252).apply(lambda x: x.prod(), raw=True) / (
    (1 + perf_returns["SPY buy&hold"]).rolling(252).apply(lambda x: x.prod(), raw=True)
) - 1
rolling_active_1y.rename("rolling 1Y active return").to_frame().plot(title="Rolling 1Y active return vs SPY")

In [ ]:
drawdown_chart(perf_returns["strategy"])

In [ ]:
active_drawdown = relative_wealth / relative_wealth.cummax() - 1
active_drawdown.rename("active drawdown vs SPY").to_frame().plot(title="Active drawdown vs SPY")

In [ ]:
heatmap_monthly(perf_returns["strategy"] - perf_returns["SPY buy&hold"])

In [ ]:
weights.tail(12).style.format("{:.1%}")